In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from scipy.stats import randint, uniform

In [32]:
import json

In [10]:
%pip install scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: C:\Users\USER\Downloads\Projects\mode_choice\jupyter-env\Scripts\python.exe -m pip install --upgrade pip


In [3]:
data = pd.read_csv('../data/loan_data.csv')

In [4]:
data.head(15)

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1
5,21.0,female,High School,12951.0,0,OWN,2500.0,VENTURE,7.14,0.19,2.0,532,No,1
6,26.0,female,Bachelor,93471.0,1,RENT,35000.0,EDUCATION,12.42,0.37,3.0,701,No,1
7,24.0,female,High School,95550.0,5,RENT,35000.0,MEDICAL,11.11,0.37,4.0,585,No,1
8,24.0,female,Associate,100684.0,3,RENT,35000.0,PERSONAL,8.90,0.35,2.0,544,No,1
9,21.0,female,High School,12739.0,0,OWN,1600.0,VENTURE,14.74,0.13,3.0,640,No,1


In [5]:
data['loan_status'].value_counts()

loan_status
0    35000
1    10000
Name: count, dtype: int64

In [21]:
def categorize_and_split(df : pd.DataFrame):

    X = df.drop('loan_status', axis = 1)
    y = df['loan_status']

    categorical_features = X.select_dtypes(
    include = ["object", "category", "bool"]).columns

    X[categorical_features] = X[categorical_features].astype('category')

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify=y)

    return X_train, X_test, y_train, y_test
    

In [7]:
def scale_weights(y):
    neg = (y==0).sum()
    pos = (y==1).sum()
    weight = neg/pos

    return weight

In [8]:
#def undersample_imbalanced(X_train, y_train):

   # rus = RandomUnderSampler(sampling_strategy =2/3)
   # X_train_res, y_train_res = rus.fit_resample(X_train, y_train)

   # print("Before:", y_train.value_counts().to_dict())
   # print("After:", y_train_res.value_counts().to_dict())

   # return X_train_res, y_train_res
    

In [28]:
def search_and_train(X, y, scale_weights):

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, stratify = y, random_state=42)

    weight = scale_weights(y_train)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    param_dist = {
        'n_estimators' : randint(100, 500),
        'max_depth': randint(3, 10),
        'min_child_weight' : randint(1, 20),
        'learning_rate': uniform(0.01, 0.3),
        'subsample': uniform(0.6 , 0.4) }
                                            

    search = RandomizedSearchCV(
        xgb.XGBClassifier(
        objective = 'binary:logistic',
        learning_rate = 0.05,
        scale_pos_weight = weight,
        tree_method = 'hist',
        enable_categorical = True
        ),

        param_distributions = param_dist,
        n_iter = 15,
        scoring = 'average_precision',
        cv = cv,
        return_train_score = True,
        random_state = 42)

    results = search.fit(X_train, y_train)

    cols = ["params", 'mean_test_score', 'std_test_score', 'mean_train_score']

    results_df = pd.DataFrame(search.cv_results_)[cols].sort_values('mean_test_score', ascending=False).reset_index(drop=True)

    best_model = search.best_estimator_

    return results_df, best_model , X_val, y_val
    

In [22]:
splitX_train, splitX_test, splity_train, splity_test = categorize_and_split(data)

In [29]:
summary_df, fit_model, X_val, y_val = search_and_train(splitX_train, splity_train, scale_weights)

In [31]:
summary_df.shape

(15, 4)

In [38]:
dtype_dict = {col :str(dtype) for col, dtype in splitX_train.dtypes.items()}

In [39]:
with open("feature_dtypes.json", "w") as f:
    json.dump(dtype_dict, f)